# QTran 第二轮完整验证（仅训练集/验证集）

本 Notebook 固定比较第一轮 Top2 与原始 QTran 对照。使用完整训练预算和五个全新开发种子，只根据验证集选择唯一配置。**整个 Notebook 不创建、不遍历、不评价测试 DataLoader。**

## 固定协议

- 候选：第一轮 Top2 + 原始配置，共3个；
- 新开发种子：`92, 102, 112, 122, 132`；
- 完整预算：每类最多2000个训练样本、60轮、耐心值10；
- 选择规则：`验证 Macro-F1 均值 - 0.5 × 样本标准差`；
- 每个任务独立保存，Kernel 中断后可续跑；
- 15/15任务全部完成前禁止冻结配置，更不得评价测试集。

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'qcs_wm811k.py').exists() and (PROJECT_DIR / 'autodl' / 'qcs_wm811k.py').exists():
    PROJECT_DIR = PROJECT_DIR / 'autodl'
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from qcs_core import environment_report
from qcs_stage2 import (
    STAGE2_SEEDS, finalize_stage2_selection, load_stage2_candidates,
    run_stage2_validation, stage2_base_config, summarize_stage2,
)

pd.set_option('display.max_columns', 100)
pd.options.display.float_format = '{:.6f}'.format
print('PROJECT_DIR:', PROJECT_DIR)
print(environment_report())

/root/xxx/autodl/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_DIR: /root/xxx/autodl
{'torch': '2.7.1+cu118', 'deepquantum': '4.5.0', 'cuda_available': True, 'cuda_device': 'NVIDIA GeForce RTX 4090'}


## 1. 路径与第一轮结果检查

In [2]:
CACHE_PATH = PROJECT_DIR / 'data_cache' / 'mixedwm38_32.npz'
STAGE1_DIR = PROJECT_DIR / 'artifacts' / 'stability_search_v2' / 'mixedwm38'
STAGE1_RESULTS = STAGE1_DIR / 'validation_screen_results.csv'
STAGE1_SUMMARY = STAGE1_DIR / 'validation_screen_summary.csv'
STAGE1_TOP2 = STAGE1_DIR / 'top2_candidates.csv'
STAGE2_ROOT = PROJECT_DIR / 'artifacts' / 'stability_search_stage2'
SPLIT_SEED = 4096

for path in (CACHE_PATH, STAGE1_RESULTS, STAGE1_SUMMARY, STAGE1_TOP2):
    print(path.exists(), path)
    assert path.exists(), path

stage1_results = pd.read_csv(STAGE1_RESULTS)
assert len(stage1_results) == 36, f'第一轮应为36行，实际为{len(stage1_results)}'
assert not stage1_results['test_evaluated'].astype(bool).any()
print('第一轮任务：36/36；没有测试集评价。')
print('第二轮输出：', STAGE2_ROOT / 'mixedwm38')

True /root/xxx/autodl/data_cache/mixedwm38_32.npz
True /root/xxx/autodl/artifacts/stability_search_v2/mixedwm38/validation_screen_results.csv
True /root/xxx/autodl/artifacts/stability_search_v2/mixedwm38/validation_screen_summary.csv
True /root/xxx/autodl/artifacts/stability_search_v2/mixedwm38/top2_candidates.csv
第一轮任务：36/36；没有测试集评价。
第二轮输出： /root/xxx/autodl/artifacts/stability_search_stage2/mixedwm38


## 2. 固定三个候选

候选从第一轮 `top2_candidates.csv` 读取，并自动加入原始对照。运行任务后，第一轮 Top2 文件的 SHA-256 会写入第二轮不可变清单。

In [3]:
candidates = load_stage2_candidates(STAGE1_TOP2)
display(candidates)
assert len(candidates) == 3
assert candidates['candidate_id'].nunique() == 3

,candidate_id,source,stage1_rank,stage1_config_id,stage1_selection_score,quantum_init_scale,quantum_lr_multiplier,quantum_pre_norm
0,stage1_rank1__init_0p020__qlr_0p500__prenorm_0,stage1_top2,1,init_0p020__qlr_0p500__prenorm_0,0.270071,0.020000,0.500000,False
1,stage1_rank2__init_0p100__qlr_0p500__prenorm_0,stage1_top2,2,init_0p100__qlr_0p500__prenorm_0,0.268684,0.100000,0.500000,False
2,original_control,original_control,0,original_init_0p100__qlr_1p000__prenorm_0,NaN,0.100000,1.000000,False


## 3. 检查完整预算

下列配置在第一次启动任务时写入 `stage2_manifest.json`，之后不允许在原目录中修改。

In [4]:
BASE_CONFIG = stage2_base_config()
print(BASE_CONFIG)
print('新开发种子:', STAGE2_SEEDS)
print('划分种子:', SPLIT_SEED)
print('总任务数:', len(candidates) * len(STAGE2_SEEDS))
assert BASE_CONFIG.epochs == 60
assert BASE_CONFIG.train_cap_per_class == 2000
assert BASE_CONFIG.eval_cap_per_class is None
assert BASE_CONFIG.quantum_projection_mode == 'five'

ExperimentConfig(image_size=32, grid_size=4, input_channels=2, stem_width=12, stem_channels=16, d_model=4, n_heads=2, n_qubits=4, quantum_depth=2, quantum_init_scale=0.1, quantum_projection_mode='five', quantum_pre_norm=False, quantum_trainable_stabilizers=False, quantum_attention_temperature=1.0, quantum_residual_scale=1.0, quantum_lr_multiplier=1.0, n_classes=9, dropout=0.15, batch_size=64, epochs=60, learning_rate=0.0005, weight_decay=0.03, label_smoothing=0.05, grad_clip=1.0, patience=10, sampler_power=0.5, train_cap_per_class=2000, eval_cap_per_class=None, num_workers=0)
新开发种子: (92, 102, 112, 122, 132)
划分种子: 4096
总任务数: 15


## 4. 查看断点续跑状态

In [5]:
required_files = ('signature.json', 'result.json', 'history.json', 'best.pt')
status_rows = []
stage2_dataset_dir = STAGE2_ROOT / 'mixedwm38'
for candidate_id in candidates['candidate_id']:
    for seed in STAGE2_SEEDS:
        job_dir = stage2_dataset_dir / candidate_id / f'seed_{seed}'
        missing = [name for name in required_files if not (job_dir / name).exists()]
        status_rows.append({
            'candidate_id': candidate_id, 'seed': seed,
            'status': '已完成' if not missing else '需要运行',
            'missing': ', '.join(missing),
        })
status = pd.DataFrame(status_rows)
display(status)
print('已完成:', int((status['status'] == '已完成').sum()), '/ 15')

,candidate_id,seed,status,missing
0,stage1_rank1__init_0p020__qlr_0p500__prenorm_0,92,需要运行,"signature.json, result.json, history.json, bes..."
1,stage1_rank1__init_0p020__qlr_0p500__prenorm_0,102,需要运行,"signature.json, result.json, history.json, bes..."
2,stage1_rank1__init_0p020__qlr_0p500__prenorm_0,112,需要运行,"signature.json, result.json, history.json, bes..."
3,stage1_rank1__init_0p020__qlr_0p500__prenorm_0,122,需要运行,"signature.json, result.json, history.json, bes..."
4,stage1_rank1__init_0p020__qlr_0p500__prenorm_0,132,需要运行,"signature.json, result.json, history.json, bes..."
5,stage1_rank2__init_0p100__qlr_0p500__prenorm_0,92,需要运行,"signature.json, result.json, history.json, bes..."
6,stage1_rank2__init_0p100__qlr_0p500__prenorm_0,102,需要运行,"signature.json, result.json, history.json, bes..."
7,stage1_rank2__init_0p100__qlr_0p500__prenorm_0,112,需要运行,"signature.json, result.json, history.json, bes..."
8,stage1_rank2__init_0p100__qlr_0p500__prenorm_0,122,需要运行,"signature.json, result.json, history.json, bes..."
9,stage1_rank2__init_0p100__qlr_0p500__prenorm_0,132,需要运行,"signature.json, result.json, history.json, bes..."


已完成: 0 / 15


## 5. 运行或续跑15个任务

`MAX_JOBS=None` 会连续完成全部剩余任务。若希望先观察稳定性，可临时设为1或3；这不会改变协议，只限制本次调用运行多少个尚未完成的任务。

In [ ]:
MAX_JOBS = None
stage2_results, candidates = run_stage2_validation(
    cache_path=CACHE_PATH,
    stage1_top2_path=STAGE1_TOP2,
    artifact_dir=STAGE2_ROOT,
    base_config=BASE_CONFIG,
    seeds=STAGE2_SEEDS,
    split_seed=SPLIT_SEED,
    resume=True,
    max_jobs=MAX_JOBS,
)
print('当前已完成:', len(stage2_results), '/ 15')
if len(stage2_results):
    assert not stage2_results['test_evaluated'].astype(bool).any()
    display(stage2_results.round(6))


[stage1_rank1__init_0p020__qlr_0p500__prenorm_0] seed=92, device=cuda



[stage1_rank1__init_0p020__qlr_0p500__prenorm_0] seed=102, device=cuda


## 6. 第二轮汇总

结果不足15行时只显示当前进度，不冻结配置。

In [ ]:
if len(stage2_results):
    stage2_summary = summarize_stage2(stage2_results)
    display(stage2_summary.round(6))
else:
    stage2_summary = pd.DataFrame()

expected_pairs = {
    (candidate_id, seed)
    for candidate_id in candidates['candidate_id']
    for seed in STAGE2_SEEDS
}
actual_pairs = (
    set(stage2_results[['candidate_id', 'seed']].itertuples(index=False, name=None))
    if len(stage2_results) else set()
)
missing_pairs = sorted(expected_pairs - actual_pairs)
print('缺少任务:', missing_pairs if missing_pairs else 'none')

## 7. 15/15完成后冻结唯一配置

本单元会生成 `frozen_stage2_selection.json`。如果任务不完整会直接报错；冻结文件已经存在但内容不一致时也会拒绝覆盖。

In [ ]:
stage2_summary, frozen_selection = finalize_stage2_selection(
    results=stage2_results,
    candidates=candidates,
    cache_path=CACHE_PATH,
    artifact_dir=STAGE2_ROOT,
    base_config=BASE_CONFIG,
    seeds=STAGE2_SEEDS,
    split_seed=SPLIT_SEED,
)
display(stage2_summary.round(6))
print('冻结候选:', frozen_selection['selected_candidate_id'])
print('冻结文件:', STAGE2_ROOT / 'mixedwm38' / 'frozen_stage2_selection.json')
print('测试指标用于选择:', frozen_selection['test_metrics_used'])

## 8. 绘制第二轮验证结果

In [ ]:
plt.rcParams.update({
    'font.family': 'Times New Roman', 'font.size': 10.5,
    'axes.titlesize': 10.5, 'axes.labelsize': 10.5,
    'xtick.labelsize': 10.5, 'ytick.labelsize': 10.5,
})
plot_table = stage2_summary.sort_values('selection_score', ascending=True)
fig, ax = plt.subplots(figsize=(10.0, 4.8))
ax.errorbar(
    plot_table['val_macro_f1_mean'], plot_table['candidate_id'],
    xerr=plot_table['val_macro_f1_std'], fmt='o', capsize=4,
)
ax.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))
ax.set_xlabel('Validation Macro-F1 (mean ± sample SD)')
ax.set_ylabel('Candidate')
ax.set_title('MixedWM38 QTran stage-2 validation-only confirmation')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
figure_path = STAGE2_ROOT / 'mixedwm38' / 'stage2_validation_macro_f1.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()
print('图片:', figure_path)

## 本阶段结束边界

第二轮只负责冻结 QTran 配置，不产生论文测试结论。下一步应为四模型等预算验证调参、数据划分审计和预先注册的最终比较。不要在本 Notebook 中添加测试评价代码，也不要根据后续测试结果返回更换本轮候选。